In [9]:
import os
import json
from pathlib import Path
from collections import OrderedDict
import xml.etree.ElementTree as ET

# ── Local Workspace Target Configuration ───────────────────────
WORKSPACE_DIR = Path("/Users/gcrane/Downloads/gemsite/classical_workspace3")
WORKSPACE_DIR.mkdir(parents=True, exist_ok=True)

NS = {'tei': 'http://www.tei-c.org/ns/1.0'}

WORK_REGISTRY = {
    "thucydides_history": {
        "textgroup": "tlg0003",
        "work": "tlg001",
        "editions": {
            "grc_jones": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0003/tlg001/tlg0003.tlg001.perseus-grc2.xml", "label": "Greek (H. S. Jones, 1942)", "class": "greek-text"},
            "eng_smith": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0003/tlg001/tlg0003.tlg001.1st1K-eng1.xml", "label": "English (C. F. Smith, 1919)", "class": "english-text"},
            "eng_crawley": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0003/tlg001/tlg0003.tlg001.perseus-eng6.xml", "label": "English (R. Crawley, 1914)", "class": "english-text"},
            "fre_betant": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0003/tlg001/tlg0003.tlg001.1st1k-fre1.xml", "label": "French (E. Bétant, 1863)", "class": "french-text"},
            "lat_haase": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0003/tlg001/tlg0003.tlg001.1st1k-lat2.xml", "label": "Latin (Fr. Haase, 1869)", "class": "latin-text"}
        }
    },
    "aristotle_poetics": {
        "textgroup": "tlg0086",
        "work": "tlg034",
        "editions": {
            "grc_kassel": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0086/tlg034/tlg0086.tlg034.perseus-grc2.xml", "label": "Greek (Kassel, 1965)", "class": "greek-text"},
            "grc_digi": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0086/tlg034/tlg0086.tlg034.digicorpus-grc2.xml", "label": "Greek (Digital Corpus Variant)", "class": "greek-text"},
            "eng_fyfe": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0086/tlg034/tlg0086.tlg034.perseus-eng2.xml", "label": "English (W.H. Fyfe, 1927)", "class": "english-text"},
            "eng_butcher": {"path": "/Users/gcrane/github/Poetics2.0/grc/tlg0086.tlg034.butcher1911-eng2.xml", "label": "English (S.H. Butcher, 1911)", "class": "english-text"},
            "eng_bywater": {"path": "/Users/gcrane/github/Poetics2.0/grc/tlg0086.tlg034.bywater1909-eng1.xml", "label": "English (Ingram Bywater, 1909)", "class": "english-text"}
        }
    }
}

# ── Enhanced Formatting Extraction Engine ───────────────────────
def extract_text_recursive(elem):
    parts = []
    tag = elem.tag.replace('{http://www.tei-c.org/ns/1.0}', '')
    
    # Process opening block container nodes and structural variants
    if tag == 'l':  # Poetic verse lines
        parts.append('<div class="verse-line">')
    elif tag == 'speaker':  # Drama speaker attribution
        parts.append('<strong class="speaker-attr">')
    elif tag == 'hi':  # Inline highlighting elements
        rend = elem.get('rend', 'italic')
        parts.append(f'<span class="render-{rend}">')
    elif tag == 'quote':  # Refined blockquote handling
        q_type = elem.get('type', 'blockquote')
        parts.append(f'<blockquote class="quote-block type-{q_type}">')

    if elem.text: 
        parts.append(elem.text)
        
    for child in elem:
        child_tag = child.tag.replace('{http://www.tei-c.org/ns/1.0}', '')
        if child_tag == 'note':
            note_text = extract_text_recursive(child).strip()
            if note_text: 
                parts.append(f'<span class="note">[{note_text}]</span>')
        elif child_tag == 'lb':  # Milestone system line breaks
            parts.append('<br/>')
        else:
            parts.append(extract_text_recursive(child))
            
        if child.tail: 
            parts.append(child.tail)
            
    # Cleanly terminate tags matching active layout tracks
    if tag == 'l':
        parts.append('</div>')
    elif tag == 'speaker':
        parts.append(': </strong>')
    elif tag == 'hi':
        parts.append('</span>')
    elif tag == 'quote':
        parts.append('</blockquote>')
        
    return ''.join(parts)

def parse_hierarchical_tei(path):
    if not os.path.exists(path): return None
    tree = ET.parse(path)
    root = tree.getroot()
    body = root.find('.//tei:body', NS) or root.find('.//body')
    if body is None: return None

    data = OrderedDict()

    def walk_divisions(node, current_path):
        tag = node.tag.replace('{http://www.tei-c.org/ns/1.0}', '')
        subtype = node.get('subtype') or node.get('type')
        n_val = node.get('n')

        if tag == 'div' and subtype in ('book', 'chapter', 'section') and n_val:
            new_path = current_path + [(subtype, n_val)]
        else:
            new_path = current_path

        paragraphs = node.findall('tei:p', NS) or node.findall('p')
        if paragraphs and new_path:
            key_map = {t: v for t, v in new_path}
            bk = key_map.get('book', '1')
            ch = key_map.get('chapter', '1')
            sec = key_map.get('section', n_val or '1')

            if bk not in data: data[bk] = OrderedDict()
            if ch not in data[bk]: data[bk][ch] = OrderedDict()
            
            combined_txt = ' '.join(extract_text_recursive(p).strip() for p in paragraphs)
            if combined_txt:
                data[bk][ch][sec] = combined_txt
            return

        for child in node:
            walk_divisions(child, new_path)

    walk_divisions(body, [])
    return data

GLOBAL_STRUCTURES = {}
GLOBAL_REGISTRIES = {}

# ── Processing Execution Phase ─────────────────────────────────
for work_key, work_meta in WORK_REGISTRY.items():
    tg = work_meta["textgroup"]
    wk = work_meta["work"]
    editions = work_meta["editions"]
    
    print(f"Ingesting textual data layers for {work_key} ({tg}.{wk})...")
    
    for v_id, cfg in editions.items():
        GLOBAL_REGISTRIES[v_id] = {
            "urn": f"urn:cts:greekLit:{tg}.{wk}.{v_id}",
            "label": cfg["label"],
            "class": cfg["class"]
        }

    work_corpus = OrderedDict()
    for v_id, cfg in editions.items():
        parsed = parse_hierarchical_tei(cfg["path"])
        if parsed:
            work_corpus[v_id] = parsed

    if not work_corpus:
        print(f" [WARNING] Ingestion matrix empty for {work_key}. Skipping.")
        continue

    first_version = list(work_corpus.keys())[0]
    baseline_corpus = work_corpus[first_version]

    structure_map = OrderedDict()
    chapter_sequence = []
    for b_k, ch_v in baseline_corpus.items():
        structure_map[b_k] = list(ch_v.keys())
        for c_k in ch_v.keys():
            chapter_sequence.append({'book': b_k, 'chapter': c_k})

    GLOBAL_STRUCTURES[f"{tg}.{wk}"] = structure_map

    print(f" -> Packaging structural formatted JS chunk scripts...")
    for c_idx, coord in enumerate(chapter_sequence):
        bk_id = coord['book']
        ch_id = coord['chapter']
        
        baseline_secs = list(baseline_corpus[bk_id][ch_id].keys())
        sections_payload = OrderedDict()
        
        for sec in baseline_secs:
            sections_payload[sec] = {}
            for v_id in editions:
                ch_data = work_corpus.get(v_id, {}).get(bk_id, {}).get(ch_id, {})
                sections_payload[sec][v_id] = ch_data.get(sec, "<i>[Text division missing in alignment layer]</i>")

        passage_urn = f"urn:cts:greekLit:{tg}.{wk}:{bk_id}.{ch_id}"
        
        chapter_payload = {
            "urn": passage_urn,
            "textgroup": tg,
            "work": wk,
            "book": bk_id,
            "chapter": ch_id,
            "sections": sections_payload,
            "navigation": {
                "prev": f"urn:cts:greekLit:{tg}.{wk}:{chapter_sequence[c_idx-1]['book']}.{chapter_sequence[c_idx-1]['chapter']}" if c_idx > 0 else None,
                "next": f"urn:cts:greekLit:{tg}.{wk}:{chapter_sequence[c_idx+1]['book']}.{chapter_sequence[c_idx+1]['chapter']}" if c_idx < len(chapter_sequence) - 1 else None
            }
        }

        chunk_dir = WORKSPACE_DIR / "corpus" / tg / wk / "chunks"
        chunk_dir.mkdir(parents=True, exist_ok=True)
        
        js_wrapped = f"""/** Perseus Autonomous Text Chunk Module **/
registerWorkspaceChunk("{passage_urn}", {json.dumps(chapter_payload, indent=2, ensure_ascii=False)});
"""
        chunk_filename = f"chunk_b{bk_id}_ch{ch_id}.js"
        (chunk_dir / chunk_filename).write_text(js_wrapped, encoding='utf-8')

print("[SUCCESS] Data ingestion asset compilation complete.")

Ingesting textual data layers for thucydides_history (tlg0003.tlg001)...
 -> Packaging structural formatted JS chunk scripts...
Ingesting textual data layers for aristotle_poetics (tlg0086.tlg034)...
 -> Packaging structural formatted JS chunk scripts...
[SUCCESS] Data ingestion asset compilation complete.


In [10]:
struct_map_json = json.dumps(GLOBAL_STRUCTURES)
text_registry_json = json.dumps(GLOBAL_REGISTRIES)

INDEX_HTML_CONTENT = """<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <title>Global Core Parallel Reading Workspace</title>
    <style>
        * { margin: 0; padding: 0; box-sizing: border-box; }
        html, body {
            height: 100%; width: 100%; overflow: hidden; 
            font-family: "Palatino Linotype", "Book Antiqua", Palatino, Georgia, serif;
            font-size: 13px; color: #000; background: #fff;
        }
        a { color: #336699; text-decoration: none; }
        a:hover { text-decoration: underline; }

        #app-view-root { display: flex; flex-direction: column; height: 100vh; width: 100vw; overflow: hidden; }

        #header-container { flex-shrink: 0; background: #fff; border-bottom: 1px solid #ccc; z-index: 10; }
        #perseus-banner { background: #660000; color: #fff; padding: 6px 15px; display: flex; justify-content: space-between; align-items: center; }
        #perseus-banner h1 a { color: #fff; font-size: 16px; font-weight: bold; }
        #perseus-banner .doc-title { font-size: 11px; color: #ffccaa; font-weight: bold; }

        #nav-bar { background: #ddddcc; border-bottom: 1px solid #999988; padding: 4px 15px; font-size: 11px; font-weight: bold; }

        #browse-bar { background: #eeeeee; border-bottom: 1px solid #cccccc; padding: 6px 15px; font-size: 11px; display: flex; flex-direction: column; gap: 5px; }
        .browse-row { display: flex; align-items: flex-start; }
        .browse-row label { font-weight: bold; width: 80px; color: #444; flex-shrink: 0; padding-top: 1px; }
        .browse-items { display: flex; flex-wrap: wrap; gap: 4px; }
        .browse-items a { padding: 1px 6px; background: #e0e0d0; color: #336699; border: 1px solid #bbbb99; font-weight: bold; border-radius: 2px; cursor: pointer; }
        .browse-items a.current { background: #660000; color: #fff; border-color: #330000; }

        .browse-items a.sec-pill { background: #fff; color: #555; border: 1px solid #ccc; font-weight: normal; font-size: 10px; padding: 1px 5px; }
        .browse-items a.sec-pill.active-pill { background: #660000; color: #fff; border-color: #330000; font-weight: bold; }

        #outer-wrapper { display: flex; width: 100%; flex: 1; min-height: 0; }

        #main-container { flex: 1; display: flex; min-width: 0; height: 100%; }
        .reading-column { display: flex; flex-direction: column; flex: 1; min-width: 0; height: 100%; border-right: 1px solid #eeeeee; background: #fff; }
        .reading-column:last-child { border-right: none; }
        .cross-panel { background: #fafafa; }

        .panel-header { background: #660000; color: #fff; padding: 4px 8px; font-size: 11px; font-weight: bold; position: sticky; top: 0; z-index: 5; display: flex; justify-content: space-between; align-items: center; flex-shrink: 0; }
        .edition-select-dropdown { background: #fff; color: #333; font-size: 10px; padding: 1px 4px; border: 1px solid #ccc; border-radius: 2px; outline: none; cursor: pointer; }

        .column-content-scroll { flex: 1; overflow-y: auto; padding: 15px; }

        .section-row { border-bottom: 1px solid #f0f0f0; padding: 6px 0; display: block; scroll-margin-top: 5px; }
        .section-row.hidden-section { display: none !important; }
        .section-row .sec-num { font-weight: bold; color: #990000; margin-right: 8px; display: inline-block; width: 35px; }
        
        .greek-text { font-size: 15px; line-height: 1.7; font-family: "Gentium Plus", "Athena", serif; color: #000; }
        .english-text { font-size: 13px; line-height: 1.6; color: #222; }
        .french-text { font-size: 13px; line-height: 1.6; color: #2b2b2b; }
        .latin-text { font-size: 13px; line-height: 1.6; color: #111; font-style: italic; }

        /* Typography Formatting Matrix Layer */
        .verse-line { padding-left: 2em; text-indent: -2em; margin-bottom: 4px; font-variant-numeric: oldstyle-nums; }
        .speaker-attr { color: #445566; font-variant: small-caps; letter-spacing: 0.5px; }
        .render-italic { font-style: italic; }
        .render-bold { font-weight: bold; }
        .render-underline { text-decoration: underline; }
        .note { font-size: 11px; color: #555; background: #fdfbf7; border-left: 2px solid #880000; padding: 2px 6px; margin: 4px 0 4px 1em; display: block; }

        /* Unified Quote and Block Citation Layout Controls */
        .quote-block {
            margin: 10px 0 10px 2.5em;
            padding-left: 12px;
            border-left: 1px dashed #bbb;
            font-size: 0.95em;
            color: #333;
        }
        /* Specific rules for indented poetry/verse quotes */
        .quote-block.type-verse {
            font-style: italic;
            line-height: 1.6;
            background: #fafafa;
            padding: 6px 12px;
        }
        /* Specific rules for standard text blockquotes */
        .quote-block.type-blockquote {
            line-height: 1.55;
            color: #1a1a1a;
        }

        .viewport-footer-controls { display: flex; justify-content: space-between; align-items: center; gap: 8px; margin-top: 20px; padding-top: 10px; border-top: 1px solid #ddd; }
        .footer-group-left { display: flex; gap: 6px; }
        .action-btn { display: inline-block; background: #660000; color: #fff !important; padding: 4px 10px; font-size: 11px; font-weight: bold; border-radius: 2px; border: none; cursor: pointer; }
        .action-btn.secondary { background: #666; }
        .action-btn:hover { background: #330000; text-decoration: none; }
        .action-btn.secondary:hover { background: #444; }
    </style>
</head>
<body>

  <div id="app-view-root">
    <div id="header-container">
      <div id="perseus-banner">
        <h1><a href="#">Perseus Workspace Engine</a></h1>
        <span class="doc-title">Global CTS-URN Unified Environment</span>
      </div>
      <div id="nav-bar">
        <span id="frame-context-label">Loading system configuration modules...</span>
      </div>
      
      <div id="browse-bar">
        <div class="browse-row">
          <label>Works:</label>
          <div class="browse-items" id="work-items-container"></div>
        </div>
        <div class="browse-row">
          <label>Books:</label>
          <div class="browse-items" id="book-items-container"></div>
        </div>
        <div class="browse-row">
          <label>Chapters:</label>
          <div class="browse-items" id="chapter-items-container"></div>
        </div>
        <div class="browse-row">
          <label>Sections:</label>
          <div class="browse-items" id="section-items-container"></div>
        </div>
      </div>
    </div>

    <div id="outer-wrapper">
      <div id="main-container">
        <div class="reading-column" id="col_f">
            <div class="panel-header">
                <span>Focal Axis Text Column</span>
                <select class="edition-select-dropdown" id="select_f" onchange="updateColumnContent('f', this.value)"></select>
            </div>
            <div class="column-content-scroll" id="content_f"></div>
        </div>
        <div class="reading-column cross-panel" id="col_c1">
            <div class="panel-header">
                <span>Comparison Version 1</span>
                <select class="edition-select-dropdown" id="select_c1" onchange="updateColumnContent('c1', this.value)"></select>
            </div>
            <div class="column-content-scroll" id="content_c1"></div>
        </div>
        <div class="reading-column cross-panel" id="col_c2">
            <div class="panel-header">
                <span>Comparison Version 2</span>
                <select class="edition-select-dropdown" id="select_c2" onchange="updateColumnContent('c2', this.value)"></select>
            </div>
            <div class="column-content-scroll" id="content_c2"></div>
        </div>
      </div>
    </div>
  </div>

  <script>
    const GLOBAL_STRUCTURES = STRUCT_REPLACE;
    const TEXT_REGISTRY = REGISTRY_REPLACE;
    
    const WORKSPACE_DATA_REGISTRY = new Map();
    
    let activeWorkKey = "";
    let activeUrnContext = "";
    let activeSectionFilter = null;

    let columnEditions = {
        f: localStorage.getItem("spa_sel_f") || "",
        c1: localStorage.getItem("spa_sel_c1") || "",
        c2: localStorage.getItem("spa_sel_c2") || ""
    };

    window.registerWorkspaceChunk = function(urn, dataPayload) {
        WORKSPACE_DATA_REGISTRY.set(urn, dataPayload);
        triggerRenderLifecycle(urn);
    };

    document.addEventListener("DOMContentLoaded", function() {
        initializeRoutingFromURL();
    });

    function populateDropdownsForWork(tgId, wkId) {
        const isAristotle = (tgId === "tlg0086" && wkId === "tlg034");
        
        if (isAristotle) {
            if (!columnEditions.f.includes("kassel") && !columnEditions.f.includes("digi")) columnEditions.f = "grc_kassel";
            if (!columnEditions.c1.includes("fyfe") && !columnEditions.c1.includes("butcher")) columnEditions.c1 = "eng_fyfe";
            if (!columnEditions.c2.includes("bywater")) columnEditions.c2 = "eng_bywater";
        } else {
            if (!columnEditions.f.includes("jones")) columnEditions.f = "grc_jones";
            if (!columnEditions.c1.includes("smith")) columnEditions.c1 = "eng_smith";
            if (!columnEditions.c2.includes("crawley") && !columnEditions.c2.includes("betant")) columnEditions.c2 = "eng_crawley";
        }

        ['f', 'c1', 'c2'].forEach(prefix => {
            const selectEl = document.getElementById(`select_${prefix}`);
            selectEl.innerHTML = "";
            Object.keys(TEXT_REGISTRY).forEach(vId => {
                const matchWork = isAristotle ? (vId.includes("kassel") || vId.includes("digi") || vId.includes("fyfe") || vId.includes("butcher") || vId.includes("bywater"))
                                              : (vId.includes("jones") || vId.includes("smith") || vId.includes("crawley") || vId.includes("betant") || vId.includes("haase"));
                if (matchWork) {
                    const opt = document.createElement("option");
                    opt.value = vId;
                    opt.innerText = TEXT_REGISTRY[vId].label;
                    if(columnEditions[prefix] === vId) opt.selected = true;
                    selectEl.appendChild(opt);
                }
            });
        });
    }

    function initializeRoutingFromURL() {
        const params = new URLSearchParams(window.location.search);
        const hash = window.location.hash;
        
        activeWorkKey = params.get("w") || Object.keys(GLOBAL_STRUCTURES)[0];
        const defaultBook = Object.keys(GLOBAL_STRUCTURES[activeWorkKey])[0];
        const b = params.get("b") || defaultBook;
        const ch = params.get("ch") || GLOBAL_STRUCTURES[activeWorkKey][b][0];
        
        if (hash && hash.startsWith("#sec_")) {
            activeSectionFilter = hash.replace("#sec_", "");
        } else {
            activeSectionFilter = null;
        }

        resolveAndInjectUrn(`urn:cts:greekLit:${activeWorkKey}:${b}.${ch}`);
    }

    window.addEventListener("hashchange", function() {
        const hash = window.location.hash;
        activeSectionFilter = (hash && hash.startsWith("#sec_")) ? hash.replace("#sec_", "") : null;
        if(activeUrnContext) {
            triggerRenderLifecycle(activeUrnContext);
        }
    });

    function resolveAndInjectUrn(urn) {
        if (WORKSPACE_DATA_REGISTRY.has(urn)) {
            triggerRenderLifecycle(urn);
            return;
        }

        document.getElementById("frame-context-label").innerText = `Resolving storage path for ${urn}...`;
        
        const parts = urn.split(":");
        const workComponents = parts[3].split(".");
        const passageComponents = parts[4].split(".");
        
        const tgId = workComponents[0];
        const wkId = workComponents[1];
        const bkId = passageComponents[0];
        const chId = passageComponents[1];

        populateDropdownsForWork(tgId, wkId);

        const script = document.createElement("script");
        script.src = `corpus/${tgId}/${wkId}/chunks/chunk_b${bkId}_ch${chId}.js`;
        script.onerror = () => {
            document.getElementById("frame-context-label").innerText = `Error: Path resolution failed for URN ${urn}`;
        };
        document.head.appendChild(script);
    }

    function triggerRenderLifecycle(urn) {
        activeUrnContext = urn;
        const payload = WORKSPACE_DATA_REGISTRY.get(urn);
        
        activeWorkKey = `${payload.textgroup}.${payload.work}`;
        populateDropdownsForWork(payload.textgroup, payload.work);

        updateURLState(payload.book, payload.chapter);
        renderNavigationControls(payload);
        renderActiveContentLayers(payload);
    }

    function updateURLState(book, chapter) {
        const params = new URLSearchParams();
        params.set("w", activeWorkKey);
        params.set("b", book);
        params.set("ch", chapter);
        const hashStr = activeSectionFilter ? `#sec_${activeSectionFilter}` : "";
        window.history.replaceState(null, "", window.location.pathname + "?" + params.toString() + hashStr);
    }

    function renderNavigationControls(payload) {
        document.getElementById("frame-context-label").innerText = `Active Frame Context: ${payload.urn}`;
        
        const workContainer = document.getElementById("work-items-container");
        workContainer.innerHTML = "";
        Object.keys(GLOBAL_STRUCTURES).forEach(wKey => {
            const a = document.createElement("a");
            a.innerText = wKey === "tlg0003.tlg001" ? "Thucydides (Histories)" : "Aristotle (Poetics)";
            if(wKey === activeWorkKey) a.className = "current";
            a.onclick = () => {
                activeSectionFilter = null;
                activeWorkKey = wKey;
                const targetBook = Object.keys(GLOBAL_STRUCTURES[wKey])[0];
                const targetChapter = GLOBAL_STRUCTURES[wKey][targetBook][0];
                resolveAndInjectUrn(`urn:cts:greekLit:${wKey}:${targetBook}.${targetChapter}`);
            };
            workContainer.appendChild(a);
        });

        const bookContainer = document.getElementById("book-items-container");
        bookContainer.innerHTML = "";
        Object.keys(GLOBAL_STRUCTURES[activeWorkKey]).forEach(bk => {
            const a = document.createElement("a");
            a.innerText = bk;
            if(bk === payload.book) a.className = "current";
            a.onclick = () => {
                activeSectionFilter = null;
                resolveAndInjectUrn(`urn:cts:greekLit:${activeWorkKey}:${bk}.${GLOBAL_STRUCTURES[activeWorkKey][bk][0]}`);
            };
            bookContainer.appendChild(a);
        });

        const chapterContainer = document.getElementById("chapter-items-container");
        chapterContainer.innerHTML = "";
        GLOBAL_STRUCTURES[activeWorkKey][payload.book].forEach(ch => {
            const a = document.createElement("a");
            a.innerText = ch;
            if(ch === payload.chapter) a.className = "current";
            a.onclick = () => {
                activeSectionFilter = null;
                resolveAndInjectUrn(`urn:cts:greekLit:${activeWorkKey}:${payload.book}.${ch}`);
            };
            chapterContainer.appendChild(a);
        });

        const sectionContainer = document.getElementById("section-items-container");
        sectionContainer.innerHTML = "";
        Object.keys(payload.sections).forEach(sec => {
            const a = document.createElement("a");
            a.innerText = sec;
            a.className = "sec-pill";
            if(sec === activeSectionFilter) a.classList.add("active-pill");
            a.href = `#sec_${sec}`;
            sectionContainer.appendChild(a);
        });
    }

    function renderActiveContentLayers(payload) {
        ['f', 'c1', 'c2'].forEach(prefix => {
            const targetContainer = document.getElementById(`content_${prefix}`);
            targetContainer.innerHTML = "";
            
            const vId = columnEditions[prefix];
            const cssClass = TEXT_REGISTRY[vId] ? TEXT_REGISTRY[vId].class : "english-text";
            
            Object.keys(payload.sections).forEach(sec => {
                const isHidden = activeSectionFilter !== null && activeSectionFilter !== sec;
                
                const row = document.createElement("div");
                row.className = `section-row s-idx-${sec} ${isHidden ? 'hidden-section' : ''}`;
                
                const txt = payload.sections[sec][vId] || "<i>[Text division missing in alignment layer]</i>";
                row.innerHTML = `
                    <span class="sec-num"><a href="#sec_${sec}">[${sec}]</a></span>
                    <span class="text-p ${cssClass}">${txt}</span>
                `;
                targetContainer.appendChild(row);
            });

            const footer = document.createElement("div");
            footer.className = "viewport-footer-controls";
            
            let prevBtnHtml = "";
            let nextBtnHtml = "";
            
            if (activeSectionFilter) {
                const secArray = Object.keys(payload.sections);
                const currentIdx = secArray.indexOf(activeSectionFilter);
                
                if (currentIdx > 0) {
                    prevBtnHtml = `<a href="#sec_${secArray[currentIdx-1]}" class="action-btn">&larr; Previous Section [${secArray[currentIdx-1]}]</a>`;
                } else if (payload.navigation.prev) {
                    prevBtnHtml = `<a class="action-btn" onclick="loadAdjacentUrn('${payload.navigation.prev}', true)">&larr; Previous Chapter</a>`;
                }
                
                if (currentIdx < secArray.length - 1) {
                    nextBtnHtml = `<a href="#sec_${secArray[currentIdx+1]}" class="action-btn">Next Section [${secArray[currentIdx+1]}] &rarr;</a>`;
                } else if (payload.navigation.next) {
                    nextBtnHtml = `<a class="action-btn" onclick="loadAdjacentUrn('${payload.navigation.next}', false)">Next Chapter &rarr;</a>`;
                }
            } else {
                if (payload.navigation.prev) {
                    prevBtnHtml = `<a class="action-btn" onclick="loadAdjacentUrn('${payload.navigation.prev}')">&larr; Previous Chapter</a>`;
                }
                if (payload.navigation.next) {
                    nextBtnHtml = `<a class="action-btn" onclick="loadAdjacentUrn('${payload.navigation.next}')">Next Chapter &rarr;</a>`;
                }
            }

            footer.innerHTML = `
                <div class="footer-group-left">
                    ${prevBtnHtml}
                    <a href="#" class="action-btn secondary" onclick="clearSectionFilter(event)">Full Chapter</a>
                </div>
                ${nextBtnHtml}
            `;
            targetContainer.appendChild(footer);
        });
    }

    window.loadAdjacentUrn = function(urn, selectLastSection = false) {
        activeSectionFilter = null;
        if (selectLastSection) {
            if (WORKSPACE_DATA_REGISTRY.has(urn)) {
                const data = WORKSPACE_DATA_REGISTRY.get(urn);
                const secs = Object.keys(data.sections);
                activeSectionFilter = secs.length > 0 ? secs[secs.length - 1] : null;
                triggerRenderLifecycle(urn);
            } else {
                const parts = urn.split(":");
                const workComponents = parts[3].split(".");
                const passageComponents = parts[4].split(".");
                const script = document.createElement("script");
                script.src = `corpus/${workComponents[0]}/${workComponents[1]}/chunks/chunk_b${passageComponents[0]}_ch${passageComponents[1]}.js`;
                script.onload = () => {
                    const data = WORKSPACE_DATA_REGISTRY.get(urn);
                    const secs = Object.keys(data.sections);
                    activeSectionFilter = secs.length > 0 ? secs[secs.length - 1] : null;
                    triggerRenderLifecycle(urn);
                };
                document.head.appendChild(script);
            }
        } else {
            resolveAndInjectUrn(urn);
        }
    };

    window.clearSectionFilter = function(e) {
        e.preventDefault();
        window.location.hash = "";
    };

    window.updateColumnContent = function(prefix, value) {
        columnEditions[prefix] = value;
        localStorage.setItem(`spa_sel_${prefix}`, value);
        if (activeUrnContext) {
            renderActiveContentLayers(WORKSPACE_DATA_REGISTRY.get(activeUrnContext));
        }
    };
  </script>
</body>
</html>
""".replace("STRUCT_REPLACE", struct_map_json).replace("REGISTRY_REPLACE", text_registry_json)

(WORKSPACE_DIR / "index.html").write_text(INDEX_HTML_CONTENT, encoding='utf-8')
print("[SUCCESS] Core dashboard frontend compiled inside classical_workspace3.")

[SUCCESS] Core dashboard frontend compiled inside classical_workspace3.
